# PARSeq training on Colab

This notebook is set up for your current PARSeq fine-tuning command, but with Colab/Drive paths and GPU training.

It does four main things:

1. Mounts Google Drive.
2. Clones the stock `baudm/parseq` repo.
3. Installs the PARSeq training dependencies.
4. Replaces the repo's `train.py` and `strhub/models/base.py` with your attached modified versions.

**Expected Drive dataset layout**

Put your LMDB data in:

```text
/content/drive/MyDrive/Parseq/Training/data/
├── train/
│   └── real/
│       └── your_train_dataset/
│           ├── data.mdb
│           └── lock.mdb
└── val/
    └── your_val_dataset/
        ├── data.mdb
        └── lock.mdb
```

The dataset subfolder names are flexible. PARSeq recursively searches for `data.mdb`.

**Runtime setting**

In Colab, use:

```text
Runtime → Change runtime type → T4 GPU
```

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Configure paths and training options

from pathlib import Path

# Edit this if you want a different Drive folder.
DRIVE_TRAINING_DIR_STR = "/content/drive/MyDrive/Parseq/Training"  #@param {type:"string"}
DRIVE_TRAINING_DIR = Path(DRIVE_TRAINING_DIR_STR)

# Your Drive dataset folder. Expected to contain train/ and val/.
DATA_SOURCE_DIR = DRIVE_TRAINING_DIR / "data"

# Local copies are much faster than training directly from Drive.
WORK_DATA_DIR = Path("/content/parseq_data")

# Where checkpoints/TensorBoard logs should be saved.
RUNS_DIR = DRIVE_TRAINING_DIR / "runs"

# PARSeq repo location in the Colab VM.
PARSEQ_DIR = Path("/content/parseq")

# Current settings based on your Mac command.
CHARSET = "94_full"              # stock PARSeq charset
MAX_LABEL_LENGTH = 44
BATCH_SIZE = 32
MAX_EPOCHS = 200
NUM_WORKERS = 2
LOG_EVERY_N_STEPS = 5

# With stock 94_full, strict should succeed. "auto" keeps your custom fallback available.
# Choices supported by your custom train.py: auto, strict, compatible.
PRETRAINED_LOAD = "auto"

# Filled in after dependencies are installed and torch is imported.
ACCELERATOR = "auto"
DEVICES = 1

print("Drive training dir:", DRIVE_TRAINING_DIR)
print("Data source dir:   ", DATA_SOURCE_DIR)
print("Local data dir:    ", WORK_DATA_DIR)
print("Runs dir:          ", RUNS_DIR)
print("PARSeq dir:        ", PARSEQ_DIR)

In [ ]:
# Check whether Colab assigned a GPU.
# This does not import torch yet, so dependency installs can happen before Python loads NumPy/Torch.
!nvidia-smi || true

## Install PARSeq and dependencies

This uses the dependency pins from your local setup where they matter most for your workflow:

- `numpy==1.26.4`
- `opencv-python-headless==4.10.0.84`
- `imgaug==0.4.0`

Colab already comes with a CUDA PyTorch build, so this notebook **does not reinstall torch/torchvision** by default.

In [ ]:
%%bash
set -euo pipefail

python -m pip install --upgrade -q pip setuptools wheel

echo "Installing pinned NumPy/OpenCV stack..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84"

echo "Installing PARSeq training dependencies..."
python -m pip install -q \
  "pytorch-lightning>=2.2,<3" \
  "hydra-core>=1.3,<1.4" \
  "omegaconf>=2.3,<2.4" \
  "torchmetrics>=1.3,<2" \
  "tensorboard" \
  "timm" \
  "einops" \
  "nltk" \
  "lmdb" \
  "fire" \
  "pillow" \
  "matplotlib" \
  "pandas" \
  "tqdm" \
  "rich"

echo "Installing imgaug with compatible pinned dependencies..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84" \
  "scipy==1.11.4" \
  "scikit-image==0.22.0" \
  "imageio==2.34.2" \
  "tifffile==2024.8.30" \
  "Shapely==2.0.6" \
  "lazy-loader==0.4"

python -m pip install -q --no-deps "imgaug==0.4.0"

echo "Re-pinning NumPy/OpenCV after dependency installs..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84"

In [ ]:
# Clone PARSeq.
import shutil
from pathlib import Path

if PARSEQ_DIR.exists():
    print(f"Removing existing repo at {PARSEQ_DIR}")
    shutil.rmtree(PARSEQ_DIR)

!git clone --depth 1 https://github.com/baudm/parseq.git "{PARSEQ_DIR}"
!python -m pip install -q -e "{PARSEQ_DIR}"

In [ ]:
# Write your modified train.py and base.py into the cloned PARSeq repo.
# These are embedded from the files attached to the ChatGPT conversation.

import base64
from pathlib import Path

PARSEQ_DIR = Path("/content/parseq")

patched_files = {
    "train.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwojIFNjZW5lIFRleHQgUmVjb2duaXRpb24gTW9kZWwgSHViCiMgQ29weXJpZ2h0IDIwMjIgRGFyd2luIEJhdXRpc3RhCiMKIyBMaWNlbnNlZCB1bmRlciB0aGUgQXBhY2hlIExpY2Vuc2UsIFZlcnNpb24gMi4wICh0aGUgIkxpY2Vuc2UiKTsKIyB5b3UgbWF5IG5vdCB1c2UgdGhpcyBmaWxlIGV4Y2VwdCBpbiBjb21wbGlhbmNlIHdpdGggdGhlIExpY2Vuc2UuCiMgWW91IG1heSBvYnRhaW4gYSBjb3B5IG9mIHRoZSBMaWNlbnNlIGF0CiMKIyAgICAgaHR0cHM6Ly93d3cuYXBhY2hlLm9yZy9saWNlbnNlcy9MSUNFTlNFLTIuMAojCiMgVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzb2Z0d2FyZQojIGRpc3RyaWJ1dGVkIHVuZGVyIHRoZSBMaWNlbnNlIGlzIGRpc3RyaWJ1dGVkIG9uIGFuICJBUyBJUyIgQkFTSVMsCiMgV0lUSE9VVCBXQVJSQU5USUVTIE9SIENPTkRJVElPTlMgT0YgQU5ZIEtJTkQsIGVpdGhlciBleHByZXNzIG9yIGltcGxpZWQuCiMgU2VlIHRoZSBMaWNlbnNlIGZvciB0aGUgc3BlY2lmaWMgbGFuZ3VhZ2UgZ292ZXJuaW5nIHBlcm1pc3Npb25zIGFuZAojIGxpbWl0YXRpb25zIHVuZGVyIHRoZSBMaWNlbnNlLgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBoeWRyYQpmcm9tIGh5ZHJhLmNvcmUuaHlkcmFfY29uZmlnIGltcG9ydCBIeWRyYUNvbmZpZwpmcm9tIG9tZWdhY29uZiBpbXBvcnQgRGljdENvbmZpZywgb3Blbl9kaWN0CgppbXBvcnQgdG9yY2gKCmZyb20gcHl0b3JjaF9saWdodG5pbmcgaW1wb3J0IFRyYWluZXIKIyBTV0EgaXMgaW50ZW50aW9uYWxseSBub3QgaW1wb3J0ZWQgYnkgZGVmYXVsdCBmb3IgdGhpcyBjdXN0b20gaGFuZHdyaXRpbmcgZmluZS10dW5pbmcgc2NyaXB0LgojIFRvIGRpc2FibGUgU1dBLCBjb21tZW50IHRoaXMgaW1wb3J0Ogpmcm9tIHB5dG9yY2hfbGlnaHRuaW5nLmNhbGxiYWNrcyBpbXBvcnQgTW9kZWxDaGVja3BvaW50LCBTdG9jaGFzdGljV2VpZ2h0QXZlcmFnaW5nCmZyb20gcHl0b3JjaF9saWdodG5pbmcubG9nZ2VycyBpbXBvcnQgVGVuc29yQm9hcmRMb2dnZXIKZnJvbSBweXRvcmNoX2xpZ2h0bmluZy5zdHJhdGVnaWVzIGltcG9ydCBERFBTdHJhdGVneQpmcm9tIHB5dG9yY2hfbGlnaHRuaW5nLnV0aWxpdGllcy5tb2RlbF9zdW1tYXJ5IGltcG9ydCBzdW1tYXJpemUKCmZyb20gc3RyaHViLmRhdGEubW9kdWxlIGltcG9ydCBTY2VuZVRleHREYXRhTW9kdWxlCmZyb20gc3RyaHViLm1vZGVscy5iYXNlIGltcG9ydCBCYXNlU3lzdGVtCmZyb20gc3RyaHViLm1vZGVscy51dGlscyBpbXBvcnQgZ2V0X3ByZXRyYWluZWRfd2VpZ2h0cwoKCiMgQ29waWVkIGZyb20gT25lQ3ljbGVMUgpkZWYgX2FubmVhbGluZ19jb3Moc3RhcnQsIGVuZCwgcGN0KToKICAgICdDb3NpbmUgYW5uZWFsIGZyb20gYHN0YXJ0YCB0byBgZW5kYCBhcyBwY3QgZ29lcyBmcm9tIDAuMCB0byAxLjAuJwogICAgY29zX291dCA9IG1hdGguY29zKG1hdGgucGkgKiBwY3QpICsgMQogICAgcmV0dXJuIGVuZCArIChzdGFydCAtIGVuZCkgLyAyLjAgKiBjb3Nfb3V0CgoKZGVmIGdldF9zd2FfbHJfZmFjdG9yKHdhcm11cF9wY3QsIHN3YV9lcG9jaF9zdGFydCwgZGl2X2ZhY3Rvcj0yNSwgZmluYWxfZGl2X2ZhY3Rvcj0xZTQpIC0+IGZsb2F0OgogICAgIiIiR2V0IHRoZSBTV0EgTFIgZmFjdG9yIGZvciB0aGUgZ2l2ZW4gYHN3YV9lcG9jaF9zdGFydGAuIEFzc3VtZXMgT25lQ3ljbGVMUiBTY2hlZHVsZXIuIiIiCiAgICB0b3RhbF9zdGVwcyA9IDEwMDAgICMgQ2FuIGJlIGFueXRoaW5nLiBXZSB1c2UgMTAwMCBmb3IgY29udmVuaWVuY2UuCiAgICBzdGFydF9zdGVwID0gaW50KHRvdGFsX3N0ZXBzICogd2FybXVwX3BjdCkgLSAxCiAgICBlbmRfc3RlcCA9IHRvdGFsX3N0ZXBzIC0gMQogICAgc3RlcF9udW0gPSBpbnQodG90YWxfc3RlcHMgKiBzd2FfZXBvY2hfc3RhcnQpIC0gMQogICAgcGN0ID0gKHN0ZXBfbnVtIC0gc3RhcnRfc3RlcCkgLyAoZW5kX3N0ZXAgLSBzdGFydF9zdGVwKQogICAgcmV0dXJuIF9hbm5lYWxpbmdfY29zKDEsIDEgLyAoZGl2X2ZhY3RvciAqIGZpbmFsX2Rpdl9mYWN0b3IpLCBwY3QpCgoKZGVmIGxvYWRfY29tcGF0aWJsZV9wcmV0cmFpbmVkX3dlaWdodHMobW9kdWxlLCBwcmV0cmFpbmVkOiBkaWN0KSAtPiBOb25lOgogICAgIiIiTG9hZCBvbmx5IHByZXRyYWluZWQgdGVuc29ycyB3aG9zZSBuYW1lcyBhbmQgc2hhcGVzIG1hdGNoIHRoZSBjdXJyZW50IG1vZGVsLgoKICAgIFRoaXMgaXMgdXNlZnVsIHdoZW4gdGhlIG1vZGVsIGNoYXJzZXQgaGFzIGNoYW5nZWQsIGZvciBleGFtcGxlIHdoZW4gYWRkaW5nIGEKICAgIHNwYWNlIGNoYXJhY3RlciwgTGFUZVgvbWF0aCBzeW1ib2xzLCBHcmVlayBsZXR0ZXJzLCBldGMuIENoYXJzZXQtZGVwZW5kZW50CiAgICBsYXllcnMgc3VjaCBhcyBoZWFkLiogYW5kIHRleHRfZW1iZWQuKiBtYXkgdGhlbiBoYXZlIGRpZmZlcmVudCBzaGFwZXMgYW5kCiAgICBuZWVkIHRvIGJlIHJhbmRvbWx5IGluaXRpYWxpemVkIGluc3RlYWQgb2YgbG9hZGVkIGZyb20gdGhlIHByZXRyYWluZWQgbW9kZWwuCiAgICAiIiIKICAgIGN1cnJlbnQgPSBtb2R1bGUuc3RhdGVfZGljdCgpCgogICAgY29tcGF0aWJsZSA9IHt9CiAgICBza2lwcGVkX3NoYXBlX21pc21hdGNoID0gW10KICAgIHNraXBwZWRfbWlzc2luZyA9IFtdCgogICAgZm9yIGtleSwgdmFsdWUgaW4gcHJldHJhaW5lZC5pdGVtcygpOgogICAgICAgIGlmIGtleSBub3QgaW4gY3VycmVudDoKICAgICAgICAgICAgc2tpcHBlZF9taXNzaW5nLmFwcGVuZChrZXkpCiAgICAgICAgZWxpZiB2YWx1ZS5zaGFwZSA9PSBjdXJyZW50W2tleV0uc2hhcGU6CiAgICAgICAgICAgIGNvbXBhdGlibGVba2V5XSA9IHZhbHVlCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2tpcHBlZF9zaGFwZV9taXNtYXRjaC5hcHBlbmQoCiAgICAgICAgICAgICAgICBmIntrZXl9OiBwcmV0cmFpbmVkIHt0dXBsZSh2YWx1ZS5zaGFwZSl9ICE9IGN1cnJlbnQge3R1cGxlKGN1cnJlbnRba2V5XS5zaGFwZSl9IgogICAgICAgICAgICApCgogICAgaWYgbm90IGNvbXBhdGlibGU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAiTm8gY29tcGF0aWJsZSBwcmV0cmFpbmVkIHRlbnNvcnMgd2VyZSBmb3VuZC4gQ2hlY2sgdGhhdCB0aGUgcHJldHJhaW5lZCAiCiAgICAgICAgICAgICJjaGVja3BvaW50IG1hdGNoZXMgdGhlIHNlbGVjdGVkIG1vZGVsIGFyY2hpdGVjdHVyZS4iCiAgICAgICAgKQoKICAgIHByaW50KGYiQ29tcGF0aWJsZSBwcmV0cmFpbmVkIGxvYWQ6IGxvYWRpbmcge2xlbihjb21wYXRpYmxlKX0ve2xlbihwcmV0cmFpbmVkKX0gdGVuc29ycy4iKQogICAgaWYgc2tpcHBlZF9zaGFwZV9taXNtYXRjaDoKICAgICAgICBwcmludCgiU2tpcHBlZCBwcmV0cmFpbmVkIGxheWVycyBkdWUgdG8gc2hhcGUgbWlzbWF0Y2g6IikKICAgICAgICBmb3IgaXRlbSBpbiBza2lwcGVkX3NoYXBlX21pc21hdGNoOgogICAgICAgICAgICBwcmludChmIiAgLSB7aXRlbX0iKQogICAgaWYgc2tpcHBlZF9taXNzaW5nOgogICAgICAgIHByaW50KCJTa2lwcGVkIHByZXRyYWluZWQgbGF5ZXJzIG5vdCBwcmVzZW50IGluIGN1cnJlbnQgbW9kZWw6IikKICAgICAgICBmb3IgaXRlbSBpbiBza2lwcGVkX21pc3Npbmc6CiAgICAgICAgICAgIHByaW50KGYiICAtIHtpdGVtfSIpCgogICAgbW9kdWxlLmxvYWRfc3RhdGVfZGljdChjb21wYXRpYmxlLCBzdHJpY3Q9RmFsc2UpCgoKQGh5ZHJhLm1haW4oY29uZmlnX3BhdGg9J2NvbmZpZ3MnLCBjb25maWdfbmFtZT0nbWFpbicsIHZlcnNpb25fYmFzZT0nMS4yJykKZGVmIG1haW4oY29uZmlnOiBEaWN0Q29uZmlnKToKICAgIHRyYWluZXJfc3RyYXRlZ3kgPSAnYXV0bycKICAgIHdpdGggb3Blbl9kaWN0KGNvbmZpZyk6CiAgICAgICAgIyBSZXNvbHZlIGFic29sdXRlIHBhdGggdG8gZGF0YS5yb290X2RpcgogICAgICAgIGNvbmZpZy5kYXRhLnJvb3RfZGlyID0gaHlkcmEudXRpbHMudG9fYWJzb2x1dGVfcGF0aChjb25maWcuZGF0YS5yb290X2RpcikKICAgICAgICAjIFNwZWNpYWwgaGFuZGxpbmcgZm9yIEdQVS1hZmZlY3RlZCBjb25maWcKICAgICAgICBncHUgPSBjb25maWcudHJhaW5lci5nZXQoJ2FjY2VsZXJhdG9yJykgPT0gJ2dwdScKICAgICAgICBkZXZpY2VzID0gY29uZmlnLnRyYWluZXIuZ2V0KCdkZXZpY2VzJywgMCkKICAgICAgICBpZiBncHU6CiAgICAgICAgICAgICMgVXNlIG1peGVkLXByZWNpc2lvbiB0cmFpbmluZwogICAgICAgICAgICBjb25maWcudHJhaW5lci5wcmVjaXNpb24gPSAnYmYxNi1taXhlZCcgaWYgdG9yY2guZ2V0X2F1dG9jYXN0X2dwdV9kdHlwZSgpIGlzIHRvcmNoLmJmbG9hdDE2IGVsc2UgJzE2LW1peGVkJwogICAgICAgIGlmIGdwdSBhbmQgZGV2aWNlcyA+IDE6CiAgICAgICAgICAgICMgVXNlIEREUCB3aXRoIG9wdGltaXphdGlvbnMKICAgICAgICAgICAgdHJhaW5lcl9zdHJhdGVneSA9IEREUFN0cmF0ZWd5KGZpbmRfdW51c2VkX3BhcmFtZXRlcnM9RmFsc2UsIGdyYWRpZW50X2FzX2J1Y2tldF92aWV3PVRydWUpCiAgICAgICAgICAgICMgU2NhbGUgc3RlcHMtYmFzZWQgY29uZmlnCiAgICAgICAgICAgIGNvbmZpZy50cmFpbmVyLnZhbF9jaGVja19pbnRlcnZhbCAvLz0gZGV2aWNlcwogICAgICAgICAgICBpZiBjb25maWcudHJhaW5lci5nZXQoJ21heF9zdGVwcycsIC0xKSA+IDA6CiAgICAgICAgICAgICAgICBjb25maWcudHJhaW5lci5tYXhfc3RlcHMgLy89IGRldmljZXMKCiAgICAjIFNwZWNpYWwgaGFuZGxpbmcgZm9yIFBBUnNlcQogICAgaWYgY29uZmlnLm1vZGVsLmdldCgncGVybV9taXJyb3JlZCcsIEZhbHNlKToKICAgICAgICBhc3NlcnQgY29uZmlnLm1vZGVsLnBlcm1fbnVtICUgMiA9PSAwLCAncGVybV9udW0gc2hvdWxkIGJlIGV2ZW4gaWYgcGVybV9taXJyb3JlZCA9IFRydWUnCgogICAgbW9kZWw6IEJhc2VTeXN0ZW0gPSBoeWRyYS51dGlscy5pbnN0YW50aWF0ZShjb25maWcubW9kZWwpCiAgICAjIElmIHNwZWNpZmllZCwgdXNlIHByZXRyYWluZWQgd2VpZ2h0cyB0byBpbml0aWFsaXplIHRoZSBtb2RlbC4KICAgICMKICAgICMgcHJldHJhaW5lZF9sb2FkIG1vZGVzOgogICAgIyAgIC0gYXV0bzogICAgICAgdHJ5IFBBUlNlcSdzIG9yaWdpbmFsIHN0cmljdCBsb2FkIGZpcnN0OyBpZiB0aGUgY2hhcnNldCBjaGFuZ2VkCiAgICAjICAgICAgICAgICAgICAgICBhbmQgY2F1c2VzIHNoYXBlIG1pc21hdGNoZXMsIGZhbGwgYmFjayB0byBjb21wYXRpYmxlIGxvYWRpbmcuCiAgICAjICAgLSBzdHJpY3Q6ICAgICBvcmlnaW5hbCBQQVJTZXEgYmVoYXZpb3I7IGJlc3Qgd2hlbiB1c2luZyB0aGUgc2FtZSBjaGFyc2V0IGFzCiAgICAjICAgICAgICAgICAgICAgICB0aGUgcHJldHJhaW5lZCBtb2RlbCwgc3VjaCBhcyB0aGUgc3RvY2sgOTRfZnVsbCBjaGFyc2V0LgogICAgIyAgIC0gY29tcGF0aWJsZTogbG9hZCBvbmx5IHRlbnNvcnMgd2hvc2UgbmFtZXMgYW5kIHNoYXBlcyBtYXRjaDsgdXNlZnVsIHdoZW4KICAgICMgICAgICAgICAgICAgICAgIGFkZGluZyBzcGFjZXMsIExhVGVYL21hdGggc3ltYm9scywgR3JlZWsgbGV0dGVycywgZXRjLgogICAgIwogICAgIyBTaW5jZSBwcmV0cmFpbmVkX2xvYWQgaXMgbm90IGluIHRoZSBzdG9jayBQQVJTZXEgY29uZmlnLCBzZXQgaXQgZnJvbSB0aGUgQ0xJIGFzCiAgICAjICtwcmV0cmFpbmVkX2xvYWQ9c3RyaWN0LCArcHJldHJhaW5lZF9sb2FkPWNvbXBhdGlibGUsIG9yICtwcmV0cmFpbmVkX2xvYWQ9YXV0by4KICAgIGlmIGNvbmZpZy5wcmV0cmFpbmVkIGlzIG5vdCBOb25lOgogICAgICAgIG0gPSBtb2RlbC5tb2RlbCBpZiBjb25maWcubW9kZWwuX3RhcmdldF8uZW5kc3dpdGgoJ1BBUlNlcScpIGVsc2UgbW9kZWwKICAgICAgICBwcmV0cmFpbmVkID0gZ2V0X3ByZXRyYWluZWRfd2VpZ2h0cyhjb25maWcucHJldHJhaW5lZCkKICAgICAgICBwcmV0cmFpbmVkX2xvYWQgPSBzdHIoY29uZmlnLmdldCgncHJldHJhaW5lZF9sb2FkJywgJ2F1dG8nKSkuc3RyaXAoKS5sb3dlcigpCgogICAgICAgIGlmIHByZXRyYWluZWRfbG9hZCA9PSAnc3RyaWN0JzoKICAgICAgICAgICAgcHJpbnQoIlByZXRyYWluZWQgbG9hZCBtb2RlOiBzdHJpY3QiKQogICAgICAgICAgICBtLmxvYWRfc3RhdGVfZGljdChwcmV0cmFpbmVkKQoKICAgICAgICBlbGlmIHByZXRyYWluZWRfbG9hZCA9PSAnY29tcGF0aWJsZSc6CiAgICAgICAgICAgIHByaW50KCJQcmV0cmFpbmVkIGxvYWQgbW9kZTogY29tcGF0aWJsZSIpCiAgICAgICAgICAgIGxvYWRfY29tcGF0aWJsZV9wcmV0cmFpbmVkX3dlaWdodHMobSwgcHJldHJhaW5lZCkKCiAgICAgICAgZWxpZiBwcmV0cmFpbmVkX2xvYWQgPT0gJ2F1dG8nOgogICAgICAgICAgICBwcmludCgiUHJldHJhaW5lZCBsb2FkIG1vZGU6IGF1dG8iKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBtLmxvYWRfc3RhdGVfZGljdChwcmV0cmFpbmVkKQogICAgICAgICAgICAgICAgcHJpbnQoIlN0cmljdCBwcmV0cmFpbmVkIGxvYWQgc3VjY2VlZGVkLiIpCiAgICAgICAgICAgIGV4Y2VwdCBSdW50aW1lRXJyb3IgYXMgZXhjOgogICAgICAgICAgICAgICAgaWYgInNpemUgbWlzbWF0Y2giIG5vdCBpbiBzdHIoZXhjKToKICAgICAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICAgICAgcHJpbnQoIlN0cmljdCBwcmV0cmFpbmVkIGxvYWQgZmFpbGVkIGR1ZSB0byBzaGFwZSBtaXNtYXRjaC4iKQogICAgICAgICAgICAgICAgcHJpbnQoIkZhbGxpbmcgYmFjayB0byBjb21wYXRpYmxlIHByZXRyYWluZWQgbG9hZGluZy4iKQogICAgICAgICAgICAgICAgbG9hZF9jb21wYXRpYmxlX3ByZXRyYWluZWRfd2VpZ2h0cyhtLCBwcmV0cmFpbmVkKQoKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJVbmtub3duIHByZXRyYWluZWRfbG9hZCBtb2RlOiB7cHJldHJhaW5lZF9sb2FkIXJ9LiAiCiAgICAgICAgICAgICAgICAiVXNlICdhdXRvJywgJ3N0cmljdCcsIG9yICdjb21wYXRpYmxlJy4iCiAgICAgICAgICAgICkKCiAgICBwcmludChzdW1tYXJpemUobW9kZWwsIG1heF9kZXB0aD0yKSkKCiAgICBkYXRhbW9kdWxlOiBTY2VuZVRleHREYXRhTW9kdWxlID0gaHlkcmEudXRpbHMuaW5zdGFudGlhdGUoY29uZmlnLmRhdGEpCgogICAgIyBTZWxlY3QgY2hlY2twb2ludHMgYnkgdmFsaWRhdGlvbiBORUQgaW5zdGVhZCBvZiBleGFjdC1tYXRjaCBhY2N1cmFjeS4gKGRhd3VkIDIwMjYtMDUtMTkpCiAgICBjaGVja3BvaW50ID0gTW9kZWxDaGVja3BvaW50KAogICAgICAgIG1vbml0b3I9J3ZhbF9ORUQnLAogICAgICAgIG1vZGU9J21heCcsCiAgICAgICAgc2F2ZV90b3Bfaz0zLAogICAgICAgIHNhdmVfbGFzdD1UcnVlLAogICAgICAgIGZpbGVuYW1lPSd7ZXBvY2h9LXtzdGVwfS17dmFsX2FjY3VyYWN5Oi40Zn0te3ZhbF9ORUQ6LjRmfScsCiAgICApCiAgICAjIGNoZWNrcG9pbnQgPSBNb2RlbENoZWNrcG9pbnQoCiAgICAjICAgICBtb25pdG9yPSd2YWxfYWNjdXJhY3knLAogICAgIyAgICAgbW9kZT0nbWF4JywKICAgICMgICAgIHNhdmVfdG9wX2s9MywKICAgICMgICAgIHNhdmVfbGFzdD1UcnVlLAogICAgIyAgICAgZmlsZW5hbWU9J3tlcG9jaH0te3N0ZXB9LXt2YWxfYWNjdXJhY3k6LjRmfS17dmFsX05FRDouNGZ9JywKICAgICMgKQoKCiAgICAjIFNXQSAvIFN0b2NoYXN0aWMgV2VpZ2h0IEF2ZXJhZ2luZyBub3RlOgogICAgIwogICAgIyBTV0EgY2FuIGJlIHVzZWZ1bCBiZWNhdXNlIGl0IGF2ZXJhZ2VzIG1vZGVsIHdlaWdodHMgZnJvbSB0aGUgbGF0ZSBwYXJ0IG9mIHRyYWluaW5nLgogICAgIyBPbiBtYW55IGxhcmdlciBvciBtb3JlIHN0YWJsZSBkYXRhc2V0cywgdGhhdCBhdmVyYWdpbmcgY2FuIGxhbmQgdGhlIG1vZGVsIGluIGEgc21vb3RoZXIKICAgICMgcmVnaW9uIG9mIHRoZSBsb3NzIHN1cmZhY2UgYW5kIGltcHJvdmUgZ2VuZXJhbGl6YXRpb24gY29tcGFyZWQgd2l0aCBvbmUgbm9pc3kgZmluYWwKICAgICMgY2hlY2twb2ludC4KICAgICMKICAgICMgRm9yIHRoaXMgc21hbGwgY3VzdG9tIGhhbmR3cml0aW5nL1BBUlNlcSBmaW5lLXR1bmluZyBzZXR1cCwgU1dBIGlzIGRpc2FibGVkIGZvciBub3cuCiAgICAjIFRoZSB2YWxpZGF0aW9uIGxvZ3Mgc2hvd2VkIHRoZSBtb2RlbCBvZnRlbiByZWFjaGVkIGl0cyBiZXN0IHJlY29nbml0aW9uIG1ldHJpY3MgYmVmb3JlCiAgICAjIHRoZSBTV0EgcGhhc2UsIHRoZW4gYmVjYW1lIG5vaXNpZXIgb3Igd29yc2UgYWZ0ZXIgTGlnaHRuaW5nIHN3YXBwZWQgT25lQ3ljbGVMUiBmb3IgU1dBTFIuCiAgICAjIFdpdGggYSBzbWFsbCB2YWxpZGF0aW9uIHNldCBhbmQgY2hhcnNldC1kZXBlbmRlbnQgb3V0cHV0IGxheWVycyBiZWluZyB0cmFpbmVkIGZyb20KICAgICMgc2NyYXRjaCwgdGhlIGF2ZXJhZ2VkIGxhdGUgd2VpZ2h0cyBjYW4gc21vb3RoIGF3YXkgYSBnb29kIHNwZWNpYWxpemVkIGNoZWNrcG9pbnQgcmF0aGVyCiAgICAjIHRoYW4gaW1wcm92ZSBpdC4KICAgICMKICAgICMgVG8gZW5hYmxlIFNXQSwgdW5jb21tZW50IHRoZSB0aHJlZSBsaW5lcyBiZWxvdywgcmVzdG9yZSB0aGUgaW1wb3J0IGFib3ZlLCBhbmQKICAgICMgY2hhbmdlIGNhbGxiYWNrcz1bY2hlY2twb2ludF0gdG8gY2FsbGJhY2tzPVtjaGVja3BvaW50LCBzd2FdLiAgKGRhd3VkLCAyMDI2LTA1LTE5KQogICAgIwogICAgc3dhX2Vwb2NoX3N0YXJ0ID0gMC43NQogICAgc3dhX2xyID0gY29uZmlnLm1vZGVsLmxyICogZ2V0X3N3YV9scl9mYWN0b3IoY29uZmlnLm1vZGVsLndhcm11cF9wY3QsIHN3YV9lcG9jaF9zdGFydCkKICAgIHN3YSA9IFN0b2NoYXN0aWNXZWlnaHRBdmVyYWdpbmcoc3dhX2xyLCBzd2FfZXBvY2hfc3RhcnQpCiAgICBjd2QgPSAoCiAgICAgICAgSHlkcmFDb25maWcuZ2V0KCkucnVudGltZS5vdXRwdXRfZGlyCiAgICAgICAgaWYgY29uZmlnLmNrcHRfcGF0aCBpcyBOb25lCiAgICAgICAgZWxzZSBzdHIoUGF0aChjb25maWcuY2twdF9wYXRoKS5wYXJlbnRzWzFdLmFic29sdXRlKCkpCiAgICApCiAgICB0cmFpbmVyOiBUcmFpbmVyID0gaHlkcmEudXRpbHMuaW5zdGFudGlhdGUoCiAgICAgICAgY29uZmlnLnRyYWluZXIsCiAgICAgICAgbG9nZ2VyPVRlbnNvckJvYXJkTG9nZ2VyKGN3ZCwgJycsICcuJyksCiAgICAgICAgc3RyYXRlZ3k9dHJhaW5lcl9zdHJhdGVneSwKICAgICAgICBlbmFibGVfbW9kZWxfc3VtbWFyeT1GYWxzZSwKICAgICAgICBjYWxsYmFja3M9W2NoZWNrcG9pbnQsIHN3YV0sCiAgICAgICAgIyBJZiBTV0EgaXMgZGlzYWJsZWQgYWJvdmUsIHVzZSB0aGlzIGluc3RlYWQ6CiAgICAgICAgIyBjYWxsYmFja3M9W2NoZWNrcG9pbnRdLAogICAgICAgIAogICAgKQogICAgdHJhaW5lci5maXQobW9kZWwsIGRhdGFtb2R1bGU9ZGF0YW1vZHVsZSwgY2twdF9wYXRoPWNvbmZpZy5ja3B0X3BhdGgpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=",
    "strhub/models/base.py": "IyBTY2VuZSBUZXh0IFJlY29nbml0aW9uIE1vZGVsIEh1YgojIENvcHlyaWdodCAyMDIyIERhcndpbiBCYXV0aXN0YQojCiMgTGljZW5zZWQgdW5kZXIgdGhlIEFwYWNoZSBMaWNlbnNlLCBWZXJzaW9uIDIuMCAodGhlICJMaWNlbnNlIik7CiMgeW91IG1heSBub3QgdXNlIHRoaXMgZmlsZSBleGNlcHQgaW4gY29tcGxpYW5jZSB3aXRoIHRoZSBMaWNlbnNlLgojIFlvdSBtYXkgb2J0YWluIGEgY29weSBvZiB0aGUgTGljZW5zZSBhdAojCiMgICAgIGh0dHBzOi8vd3d3LmFwYWNoZS5vcmcvbGljZW5zZXMvTElDRU5TRS0yLjAKIwojIFVubGVzcyByZXF1aXJlZCBieSBhcHBsaWNhYmxlIGxhdyBvciBhZ3JlZWQgdG8gaW4gd3JpdGluZywgc29mdHdhcmUKIyBkaXN0cmlidXRlZCB1bmRlciB0aGUgTGljZW5zZSBpcyBkaXN0cmlidXRlZCBvbiBhbiAiQVMgSVMiIEJBU0lTLAojIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvciBpbXBsaWVkLgojIFNlZSB0aGUgTGljZW5zZSBmb3IgdGhlIHNwZWNpZmljIGxhbmd1YWdlIGdvdmVybmluZyBwZXJtaXNzaW9ucyBhbmQKIyBsaW1pdGF0aW9ucyB1bmRlciB0aGUgTGljZW5zZS4KCmltcG9ydCBtYXRoCmZyb20gYWJjIGltcG9ydCBBQkMsIGFic3RyYWN0bWV0aG9kCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmZyb20gbmx0ayBpbXBvcnQgZWRpdF9kaXN0YW5jZQoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yCmZyb20gdG9yY2gub3B0aW0gaW1wb3J0IE9wdGltaXplcgpmcm9tIHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlciBpbXBvcnQgT25lQ3ljbGVMUgoKaW1wb3J0IHB5dG9yY2hfbGlnaHRuaW5nIGFzIHBsCmZyb20gcHl0b3JjaF9saWdodG5pbmcudXRpbGl0aWVzLnR5cGVzIGltcG9ydCBTVEVQX09VVFBVVApmcm9tIHRpbW0ub3B0aW0gaW1wb3J0IGNyZWF0ZV9vcHRpbWl6ZXJfdjIKCmZyb20gc3RyaHViLmRhdGEudXRpbHMgaW1wb3J0IEJhc2VUb2tlbml6ZXIsIENoYXJzZXRBZGFwdGVyLCBDVENUb2tlbml6ZXIsIFRva2VuaXplcgoKCkBkYXRhY2xhc3MKY2xhc3MgQmF0Y2hSZXN1bHQ6CiAgICBudW1fc2FtcGxlczogaW50CiAgICBjb3JyZWN0OiBpbnQKICAgIG5lZDogZmxvYXQKICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBsYWJlbF9sZW5ndGg6IGludAogICAgbG9zczogVGVuc29yCiAgICBsb3NzX251bWVsOiBpbnQKCgpFUE9DSF9PVVRQVVQgPSBsaXN0W2RpY3Rbc3RyLCBCYXRjaFJlc3VsdF1dCgoKY2xhc3MgQmFzZVN5c3RlbShwbC5MaWdodG5pbmdNb2R1bGUsIEFCQyk6CgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgdG9rZW5pemVyOiBCYXNlVG9rZW5pemVyLAogICAgICAgIGNoYXJzZXRfdGVzdDogc3RyLAogICAgICAgIGJhdGNoX3NpemU6IGludCwKICAgICAgICBscjogZmxvYXQsCiAgICAgICAgd2FybXVwX3BjdDogZmxvYXQsCiAgICAgICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnRva2VuaXplciA9IHRva2VuaXplcgogICAgICAgIHNlbGYuY2hhcnNldF9hZGFwdGVyID0gQ2hhcnNldEFkYXB0ZXIoY2hhcnNldF90ZXN0KQogICAgICAgIHNlbGYuYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUKICAgICAgICBzZWxmLmxyID0gbHIKICAgICAgICBzZWxmLndhcm11cF9wY3QgPSB3YXJtdXBfcGN0CiAgICAgICAgc2VsZi53ZWlnaHRfZGVjYXkgPSB3ZWlnaHRfZGVjYXkKICAgICAgICBzZWxmLm91dHB1dHM6IEVQT0NIX09VVFBVVCA9IFtdCgogICAgQGFic3RyYWN0bWV0aG9kCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZXM6IFRlbnNvciwgbWF4X2xlbmd0aDogT3B0aW9uYWxbaW50XSA9IE5vbmUpIC0+IFRlbnNvcjoKICAgICAgICAiIiJJbmZlcmVuY2UKCiAgICAgICAgQXJnczoKICAgICAgICAgICAgaW1hZ2VzOiBCYXRjaCBvZiBpbWFnZXMuIFNoYXBlOiBOLCBDaCwgSCwgVwogICAgICAgICAgICBtYXhfbGVuZ3RoOiBNYXggc2VxdWVuY2UgbGVuZ3RoIG9mIHRoZSBvdXRwdXQuIElmIE5vbmUsIHdpbGwgdXNlIGRlZmF1bHQuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIGxvZ2l0czogTiwgTCwgQyAoTCA9IHNlcXVlbmNlIGxlbmd0aCwgQyA9IG51bWJlciBvZiBjbGFzc2VzLCB0eXBpY2FsbHkgbGVuKGNoYXJzZXRfdHJhaW4pICsgbnVtIHNwZWNpYWxzKQogICAgICAgICIiIgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKCiAgICBAYWJzdHJhY3RtZXRob2QKICAgIGRlZiBmb3J3YXJkX2xvZ2l0c19sb3NzKHNlbGYsIGltYWdlczogVGVuc29yLCBsYWJlbHM6IGxpc3Rbc3RyXSkgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3IsIGludF06CiAgICAgICAgIiIiTGlrZSBmb3J3YXJkKCksIGJ1dCBhbHNvIGNvbXB1dGVzIHRoZSBsb3NzIChjYWxscyBmb3J3YXJkKCkgaW50ZXJuYWxseSkuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGltYWdlczogQmF0Y2ggb2YgaW1hZ2VzLiBTaGFwZTogTiwgQ2gsIEgsIFcKICAgICAgICAgICAgbGFiZWxzOiBUZXh0IGxhYmVscyBvZiB0aGUgaW1hZ2VzCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIGxvZ2l0czogTiwgTCwgQyAoTCA9IHNlcXVlbmNlIGxlbmd0aCwgQyA9IG51bWJlciBvZiBjbGFzc2VzLCB0eXBpY2FsbHkgbGVuKGNoYXJzZXRfdHJhaW4pICsgbnVtIHNwZWNpYWxzKQogICAgICAgICAgICBsb3NzOiBtZWFuIGxvc3MgZm9yIHRoZSBiYXRjaAogICAgICAgICAgICBsb3NzX251bWVsOiBudW1iZXIgb2YgZWxlbWVudHMgdGhlIGxvc3Mgd2FzIGNhbGN1bGF0ZWQgZnJvbQogICAgICAgICIiIgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKCiAgICBkZWYgY29uZmlndXJlX29wdGltaXplcnMoc2VsZik6CiAgICAgICAgYWdiID0gc2VsZi50cmFpbmVyLmFjY3VtdWxhdGVfZ3JhZF9iYXRjaGVzCiAgICAgICAgIyBMaW5lYXIgc2NhbGluZyBzbyB0aGF0IHRoZSBlZmZlY3RpdmUgbGVhcm5pbmcgcmF0ZSBpcyBjb25zdGFudCByZWdhcmRsZXNzIG9mIHRoZSBudW1iZXIgb2YgR1BVcyB1c2VkIHdpdGggRERQLgogICAgICAgIGxyX3NjYWxlID0gYWdiICogbWF0aC5zcXJ0KHNlbGYudHJhaW5lci5udW1fZGV2aWNlcykgKiBzZWxmLmJhdGNoX3NpemUgLyAyNTYuMAogICAgICAgIGxyID0gbHJfc2NhbGUgKiBzZWxmLmxyCiAgICAgICAgb3B0aW0gPSBjcmVhdGVfb3B0aW1pemVyX3YyKHNlbGYsICdhZGFtdycsIGxyLCBzZWxmLndlaWdodF9kZWNheSkKICAgICAgICBzY2hlZCA9IE9uZUN5Y2xlTFIoCiAgICAgICAgICAgIG9wdGltLCBsciwgc2VsZi50cmFpbmVyLmVzdGltYXRlZF9zdGVwcGluZ19iYXRjaGVzLCBwY3Rfc3RhcnQ9c2VsZi53YXJtdXBfcGN0LCBjeWNsZV9tb21lbnR1bT1GYWxzZQogICAgICAgICkKICAgICAgICByZXR1cm4geydvcHRpbWl6ZXInOiBvcHRpbSwgJ2xyX3NjaGVkdWxlcic6IHsnc2NoZWR1bGVyJzogc2NoZWQsICdpbnRlcnZhbCc6ICdzdGVwJ319CgogICAgZGVmIG9wdGltaXplcl96ZXJvX2dyYWQoc2VsZiwgZXBvY2g6IGludCwgYmF0Y2hfaWR4OiBpbnQsIG9wdGltaXplcjogT3B0aW1pemVyKSAtPiBOb25lOgogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKCiAgICBkZWYgX2V2YWxfc3RlcChzZWxmLCBiYXRjaCwgdmFsaWRhdGlvbjogYm9vbCkgLT4gT3B0aW9uYWxbU1RFUF9PVVRQVVRdOgogICAgICAgIGltYWdlcywgbGFiZWxzID0gYmF0Y2gKCiAgICAgICAgY29ycmVjdCA9IDAKICAgICAgICB0b3RhbCA9IDAKICAgICAgICBuZWQgPSAwCiAgICAgICAgY29uZmlkZW5jZSA9IDAKICAgICAgICBsYWJlbF9sZW5ndGggPSAwCiAgICAgICAgaWYgdmFsaWRhdGlvbjoKICAgICAgICAgICAgbG9naXRzLCBsb3NzLCBsb3NzX251bWVsID0gc2VsZi5mb3J3YXJkX2xvZ2l0c19sb3NzKGltYWdlcywgbGFiZWxzKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgQXQgdGVzdC10aW1lLCB3ZSBzaG91bGRuJ3Qgc3BlY2lmeSBhIG1heF9sYWJlbF9sZW5ndGggYmVjYXVzZSB0aGUgdGVzdC10aW1lIGNoYXJzZXQgdXNlZAogICAgICAgICAgICAjIG1pZ2h0IGJlIGRpZmZlcmVudCBmcm9tIHRoZSB0cmFpbi10aW1lIGNoYXJzZXQuIG1heF9sYWJlbF9sZW5ndGggaW4gZXZhbF9sb2dpdHNfbG9zcygpIGlzIGNvbXB1dGVkCiAgICAgICAgICAgICMgYmFzZWQgb24gdGhlIHRyYW5zZm9ybWVkIGxhYmVsLCB3aGljaCBjb3VsZCBiZSB3cm9uZyBpZiB0aGUgYWN0dWFsIGd0IGxhYmVsIGNvbnRhaW5zIGNoYXJhY3RlcnMgZXhpc3RpbmcKICAgICAgICAgICAgIyBpbiB0aGUgdHJhaW4tdGltZSBjaGFyc2V0IGJ1dCBub3QgaW4gdGhlIHRlc3QtdGltZSBjaGFyc2V0LiBGb3IgZXhhbXBsZSwgImFpc2hhaGFsZXllcy5ibG9nc3BvdC5jb20iCiAgICAgICAgICAgICMgaXMgZXhhY3RseSAyNSBjaGFyYWN0ZXJzLCBidXQgaWYgcHJvY2Vzc2VkIGJ5IENoYXJzZXRBZGFwdGVyIGZvciB0aGUgMzYtY2hhciBzZXQsIGl0IGJlY29tZXMgMjMgY2hhcmFjdGVycwogICAgICAgICAgICAjIGxvbmcgb25seSwgd2hpY2ggc2V0cyBtYXhfbGFiZWxfbGVuZ3RoID0gMjMuIFRoaXMgd2lsbCBjYXVzZSB0aGUgbW9kZWwgcHJlZGljdGlvbiB0byBiZSB0cnVuY2F0ZWQuCiAgICAgICAgICAgIGxvZ2l0cyA9IHNlbGYuZm9yd2FyZChpbWFnZXMpCiAgICAgICAgICAgIGxvc3MgPSBsb3NzX251bWVsID0gTm9uZSAgIyBPbmx5IHVzZWQgZm9yIHZhbGlkYXRpb247IG5vdCBuZWVkZWQgYXQgdGVzdC10aW1lLgoKICAgICAgICBwcm9icyA9IGxvZ2l0cy5zb2Z0bWF4KC0xKQogICAgICAgIHByZWRzLCBwcm9icyA9IHNlbGYudG9rZW5pemVyLmRlY29kZShwcm9icykKICAgICAgICBmb3IgcHJlZCwgcHJvYiwgZ3QgaW4gemlwKHByZWRzLCBwcm9icywgbGFiZWxzKToKICAgICAgICAgICAgY29uZmlkZW5jZSArPSBwcm9iLnByb2QoKS5pdGVtKCkKICAgICAgICAgICAgcHJlZCA9IHNlbGYuY2hhcnNldF9hZGFwdGVyKHByZWQpCiAgICAgICAgICAgICMgRm9sbG93IElDREFSIDIwMTkgZGVmaW5pdGlvbiBvZiBOLkUuRC4KICAgICAgICAgICAgbmVkICs9IGVkaXRfZGlzdGFuY2UocHJlZCwgZ3QpIC8gbWF4KGxlbihwcmVkKSwgbGVuKGd0KSkKICAgICAgICAgICAgaWYgcHJlZCA9PSBndDoKICAgICAgICAgICAgICAgIGNvcnJlY3QgKz0gMQogICAgICAgICAgICB0b3RhbCArPSAxCiAgICAgICAgICAgIGxhYmVsX2xlbmd0aCArPSBsZW4ocHJlZCkKICAgICAgICByZXR1cm4gZGljdChvdXRwdXQ9QmF0Y2hSZXN1bHQodG90YWwsIGNvcnJlY3QsIG5lZCwgY29uZmlkZW5jZSwgbGFiZWxfbGVuZ3RoLCBsb3NzLCBsb3NzX251bWVsKSkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX2FnZ3JlZ2F0ZV9yZXN1bHRzKG91dHB1dHM6IEVQT0NIX09VVFBVVCkgLT4gdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdF06CiAgICAgICAgaWYgbm90IG91dHB1dHM6CiAgICAgICAgICAgIHJldHVybiAwLjAsIDAuMCwgMC4wCiAgICAgICAgdG90YWxfbG9zcyA9IDAKICAgICAgICB0b3RhbF9sb3NzX251bWVsID0gMAogICAgICAgIHRvdGFsX25fY29ycmVjdCA9IDAKICAgICAgICB0b3RhbF9ub3JtX0VEID0gMAogICAgICAgIHRvdGFsX3NpemUgPSAwCiAgICAgICAgZm9yIHJlc3VsdCBpbiBvdXRwdXRzOgogICAgICAgICAgICByZXN1bHQgPSByZXN1bHRbJ291dHB1dCddCiAgICAgICAgICAgIHRvdGFsX2xvc3MgKz0gcmVzdWx0Lmxvc3NfbnVtZWwgKiByZXN1bHQubG9zcwogICAgICAgICAgICB0b3RhbF9sb3NzX251bWVsICs9IHJlc3VsdC5sb3NzX251bWVsCiAgICAgICAgICAgIHRvdGFsX25fY29ycmVjdCArPSByZXN1bHQuY29ycmVjdAogICAgICAgICAgICB0b3RhbF9ub3JtX0VEICs9IHJlc3VsdC5uZWQKICAgICAgICAgICAgdG90YWxfc2l6ZSArPSByZXN1bHQubnVtX3NhbXBsZXMKICAgICAgICBhY2MgPSB0b3RhbF9uX2NvcnJlY3QgLyB0b3RhbF9zaXplCiAgICAgICAgbmVkID0gMSAtIHRvdGFsX25vcm1fRUQgLyB0b3RhbF9zaXplCiAgICAgICAgbG9zcyA9IHRvdGFsX2xvc3MgLyB0b3RhbF9sb3NzX251bWVsCiAgICAgICAgcmV0dXJuIGFjYywgbmVkLCBsb3NzCgogICAgZGVmIHZhbGlkYXRpb25fc3RlcChzZWxmLCBiYXRjaCwgYmF0Y2hfaWR4KSAtPiBPcHRpb25hbFtTVEVQX09VVFBVVF06CiAgICAgICAgcmVzdWx0ID0gc2VsZi5fZXZhbF9zdGVwKGJhdGNoLCBUcnVlKQogICAgICAgIHNlbGYub3V0cHV0cy5hcHBlbmQocmVzdWx0KQogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgb25fdmFsaWRhdGlvbl9lcG9jaF9lbmQoc2VsZikgLT4gTm9uZToKICAgICAgICBhY2MsIG5lZCwgbG9zcyA9IHNlbGYuX2FnZ3JlZ2F0ZV9yZXN1bHRzKHNlbGYub3V0cHV0cykKICAgICAgICBzZWxmLm91dHB1dHMuY2xlYXIoKQoKICAgICAgICB2YWxfYWNjdXJhY3kgPSAxMDAgKiBhY2MKICAgICAgICB2YWxfbmVkID0gMTAwICogbmVkCgogICAgICAgIHNlbGYubG9nKCd2YWxfYWNjdXJhY3knLCB2YWxfYWNjdXJhY3ksIHN5bmNfZGlzdD1UcnVlKQogICAgICAgIHNlbGYubG9nKCd2YWxfTkVEJywgdmFsX25lZCwgc3luY19kaXN0PVRydWUpCiAgICAgICAgc2VsZi5sb2coJ3ZhbF9sb3NzJywgbG9zcywgc3luY19kaXN0PVRydWUpCiAgICAgICAgc2VsZi5sb2coJ2hwX21ldHJpYycsIGFjYywgc3luY19kaXN0PVRydWUpCgogICAgICAgICMgRG8gbm90IHByaW50IGR1cmluZyBMaWdodG5pbmcncyBwcmUtdHJhaW5pbmcgc2FuaXR5IHZhbGlkYXRpb24uCiAgICAgICAgaWYgZ2V0YXR0cihzZWxmLnRyYWluZXIsICJzYW5pdHlfY2hlY2tpbmciLCBGYWxzZSk6CiAgICAgICAgICAgIHJldHVybgoKICAgICAgICBsb3NzX3ZhbHVlID0gbG9zcy5kZXRhY2goKS5pdGVtKCkgaWYgdG9yY2guaXNfdGVuc29yKGxvc3MpIGVsc2UgZmxvYXQobG9zcykKCiAgICAgICAgIyBBdm9pZCBkdXBsaWNhdGUgcHJpbnRzIGlmIHVzaW5nIG11bHRpcGxlIGRldmljZXMgbGF0ZXIuCiAgICAgICAgaWYgc2VsZi50cmFpbmVyLmlzX2dsb2JhbF96ZXJvOgogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiXG5FcG9jaCB7c2VsZi5jdXJyZW50X2Vwb2NofTogIgogICAgICAgICAgICAgICAgZiJ2YWxfYWNjdXJhY3k9e3ZhbF9hY2N1cmFjeTouMmZ9JSB8ICIKICAgICAgICAgICAgICAgIGYidmFsX05FRD17dmFsX25lZDouMmZ9JSB8ICIKICAgICAgICAgICAgICAgIGYidmFsX2xvc3M9e2xvc3NfdmFsdWU6LjRmfSIsCiAgICAgICAgICAgICAgICBmbHVzaD1UcnVlLAogICAgICAgICAgICApCgogICAgZGVmIHRlc3Rfc3RlcChzZWxmLCBiYXRjaCwgYmF0Y2hfaWR4KSAtPiBPcHRpb25hbFtTVEVQX09VVFBVVF06CiAgICAgICAgcmV0dXJuIHNlbGYuX2V2YWxfc3RlcChiYXRjaCwgRmFsc2UpCgoKY2xhc3MgQ3Jvc3NFbnRyb3B5U3lzdGVtKEJhc2VTeXN0ZW0pOgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLCBjaGFyc2V0X3RyYWluOiBzdHIsIGNoYXJzZXRfdGVzdDogc3RyLCBiYXRjaF9zaXplOiBpbnQsIGxyOiBmbG9hdCwgd2FybXVwX3BjdDogZmxvYXQsIHdlaWdodF9kZWNheTogZmxvYXQKICAgICkgLT4gTm9uZToKICAgICAgICB0b2tlbml6ZXIgPSBUb2tlbml6ZXIoY2hhcnNldF90cmFpbikKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHRva2VuaXplciwgY2hhcnNldF90ZXN0LCBiYXRjaF9zaXplLCBsciwgd2FybXVwX3BjdCwgd2VpZ2h0X2RlY2F5KQogICAgICAgIHNlbGYuYm9zX2lkID0gdG9rZW5pemVyLmJvc19pZAogICAgICAgIHNlbGYuZW9zX2lkID0gdG9rZW5pemVyLmVvc19pZAogICAgICAgIHNlbGYucGFkX2lkID0gdG9rZW5pemVyLnBhZF9pZAoKICAgIGRlZiBmb3J3YXJkX2xvZ2l0c19sb3NzKHNlbGYsIGltYWdlczogVGVuc29yLCBsYWJlbHM6IGxpc3Rbc3RyXSkgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3IsIGludF06CiAgICAgICAgdGFyZ2V0cyA9IHNlbGYudG9rZW5pemVyLmVuY29kZShsYWJlbHMsIHNlbGYuZGV2aWNlKQogICAgICAgIHRhcmdldHMgPSB0YXJnZXRzWzosIDE6XSAgIyBEaXNjYXJkIDxib3M+CiAgICAgICAgbWF4X2xlbiA9IHRhcmdldHMuc2hhcGVbMV0gLSAxICAjIGV4Y2x1ZGUgPGVvcz4gZnJvbSBjb3VudAogICAgICAgIGxvZ2l0cyA9IHNlbGYuZm9yd2FyZChpbWFnZXMsIG1heF9sZW4pCiAgICAgICAgbG9zcyA9IEYuY3Jvc3NfZW50cm9weShsb2dpdHMuZmxhdHRlbihlbmRfZGltPTEpLCB0YXJnZXRzLmZsYXR0ZW4oKSwgaWdub3JlX2luZGV4PXNlbGYucGFkX2lkKQogICAgICAgIGxvc3NfbnVtZWwgPSAodGFyZ2V0cyAhPSBzZWxmLnBhZF9pZCkuc3VtKCkKICAgICAgICByZXR1cm4gbG9naXRzLCBsb3NzLCBsb3NzX251bWVsCgoKY2xhc3MgQ1RDU3lzdGVtKEJhc2VTeXN0ZW0pOgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLCBjaGFyc2V0X3RyYWluOiBzdHIsIGNoYXJzZXRfdGVzdDogc3RyLCBiYXRjaF9zaXplOiBpbnQsIGxyOiBmbG9hdCwgd2FybXVwX3BjdDogZmxvYXQsIHdlaWdodF9kZWNheTogZmxvYXQKICAgICkgLT4gTm9uZToKICAgICAgICB0b2tlbml6ZXIgPSBDVENUb2tlbml6ZXIoY2hhcnNldF90cmFpbikKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHRva2VuaXplciwgY2hhcnNldF90ZXN0LCBiYXRjaF9zaXplLCBsciwgd2FybXVwX3BjdCwgd2VpZ2h0X2RlY2F5KQogICAgICAgIHNlbGYuYmxhbmtfaWQgPSB0b2tlbml6ZXIuYmxhbmtfaWQKCiAgICBkZWYgZm9yd2FyZF9sb2dpdHNfbG9zcyhzZWxmLCBpbWFnZXM6IFRlbnNvciwgbGFiZWxzOiBsaXN0W3N0cl0pIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yLCBpbnRdOgogICAgICAgIHRhcmdldHMgPSBzZWxmLnRva2VuaXplci5lbmNvZGUobGFiZWxzLCBzZWxmLmRldmljZSkKICAgICAgICBsb2dpdHMgPSBzZWxmLmZvcndhcmQoaW1hZ2VzKQogICAgICAgIGxvZ19wcm9icyA9IGxvZ2l0cy5sb2dfc29mdG1heCgtMSkudHJhbnNwb3NlKDAsIDEpICAjIHN3YXAgYmF0Y2ggYW5kIHNlcS4gZGltcwogICAgICAgIFQsIE4sIF8gPSBsb2dfcHJvYnMuc2hhcGUKICAgICAgICBpbnB1dF9sZW5ndGhzID0gdG9yY2guZnVsbChzaXplPShOLCksIGZpbGxfdmFsdWU9VCwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXNlbGYuZGV2aWNlKQogICAgICAgIHRhcmdldF9sZW5ndGhzID0gdG9yY2guYXNfdGVuc29yKGxpc3QobWFwKGxlbiwgbGFiZWxzKSksIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBsb3NzID0gRi5jdGNfbG9zcyhsb2dfcHJvYnMsIHRhcmdldHMsIGlucHV0X2xlbmd0aHMsIHRhcmdldF9sZW5ndGhzLCBibGFuaz1zZWxmLmJsYW5rX2lkLCB6ZXJvX2luZmluaXR5PVRydWUpCiAgICAgICAgcmV0dXJuIGxvZ2l0cywgbG9zcywgTgo=",
}

for rel_path, b64_text in patched_files.items():
    target = PARSEQ_DIR / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(base64.b64decode(b64_text))
    print("Wrote", target)

print("\nSanity checks:")
!grep -n "pretrained_load\|val_NED\|callbacks=\[checkpoint" /content/parseq/train.py | head -50
!grep -n "val_accuracy=.*val_NED\|val_loss" /content/parseq/strhub/models/base.py | head -50

In [ ]:
# Verify installed versions/imports, then choose GPU/CPU training.
import numpy, cv2, torch, imgaug
import pytorch_lightning, hydra, omegaconf, timm, lmdb

print("NumPy:             ", numpy.__version__)
print("OpenCV:            ", cv2.__version__)
print("Torch:             ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:               ", torch.cuda.get_device_name(0))
print("imgaug:            ", imgaug.__version__)
print("pytorch_lightning: ", pytorch_lightning.__version__)
print("hydra:             ", hydra.__version__)
print("omegaconf:         ", omegaconf.__version__)
print("timm:              ", timm.__version__)
print("lmdb OK")

assert numpy.__version__ == "1.26.4", "NumPy version is not pinned to 1.26.4"
assert cv2.__version__ == "4.10.0", "OpenCV version is not pinned to 4.10.0"
assert imgaug.__version__ == "0.4.0", "imgaug version is not 0.4.0"

ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1
print("Training accelerator:", ACCELERATOR)

## Copy the LMDB dataset from Drive to local Colab storage

Training from `/content` is usually faster and less flaky than training directly from Google Drive.

This cell copies:

```text
/content/drive/MyDrive/Parseq/Training/data
```

to:

```text
/content/parseq_data
```

In [ ]:
# Copy dataset from Drive to local runtime storage.
import os, shutil, subprocess
from pathlib import Path

if not DATA_SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"DATA_SOURCE_DIR does not exist: {DATA_SOURCE_DIR}\n"
        "Edit DRIVE_TRAINING_DIR or DATA_SOURCE_DIR in the config cell."
    )

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)

WORK_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

src = str(DATA_SOURCE_DIR) + "/"
dst = str(WORK_DATA_DIR) + "/"

print(f"Copying dataset:\n  {src}\n→ {dst}")

try:
    subprocess.run(["rsync", "-a", "--info=progress2", src, dst], check=True)
except Exception as exc:
    print("rsync failed; falling back to shutil.copytree:", exc)
    shutil.copytree(DATA_SOURCE_DIR, WORK_DATA_DIR)

print("\nLocal dataset folders:")
!find "{WORK_DATA_DIR}" -maxdepth 4 -type f \( -name "data.mdb" -o -name "lock.mdb" \) -print

## Optional: inspect LMDB counts and label problems

This catches the common reasons PARSeq silently trains/evaluates on fewer samples than expected:

- labels longer than `model.max_label_length`
- labels containing characters outside the selected charset
- missing/incorrect LMDB structure

Because `data.remove_whitespace=false`, spaces are preserved and counted.

In [ ]:
# Inspect LMDB sample counts, label lengths, and charset compatibility.
from pathlib import Path
from omegaconf import OmegaConf
import lmdb
from collections import Counter

charset_cfg_path = PARSEQ_DIR / "configs" / "charset" / f"{CHARSET}.yaml"
if not charset_cfg_path.exists():
    raise FileNotFoundError(f"Could not find charset config: {charset_cfg_path}")

cfg = OmegaConf.load(charset_cfg_path)
charset_train = str(cfg.model.charset_train)
allowed = set(charset_train)

print("Charset:", CHARSET)
print("charset_train length:", len(charset_train))
print("charset_train repr:", repr(charset_train))
print("MAX_LABEL_LENGTH:", MAX_LABEL_LENGTH)

def read_lmdb_labels(db_file: Path):
    env = lmdb.open(
        str(db_file.parent),
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
        max_readers=1,
    )
    labels = []
    with env.begin(write=False) as txn:
        raw_n = txn.get(b"num-samples")
        if raw_n is None:
            # Fallback: scan labels if num-samples is missing.
            with txn.cursor() as cur:
                for key, value in cur:
                    if key.startswith(b"label-"):
                        labels.append(value.decode("utf-8", errors="replace"))
            return labels

        n = int(raw_n.decode())
        for i in range(1, n + 1):
            raw = txn.get(f"label-{i:09d}".encode())
            if raw is None:
                labels.append(None)
            else:
                labels.append(raw.decode("utf-8", errors="replace"))
    env.close()
    return labels

def inspect_split(split_name: str, root: Path):
    dbs = sorted(root.rglob("data.mdb"))
    print(f"\n=== {split_name} ===")
    if not dbs:
        print("No data.mdb files found under", root)
        return

    total = 0
    missing = 0
    too_long = []
    bad_charset = []

    for db in dbs:
        labels = read_lmdb_labels(db)
        total += len(labels)
        print(f"{db.parent}: {len(labels)} samples")
        for idx, label in enumerate(labels, start=1):
            if label is None:
                missing += 1
                continue
            if len(label) > MAX_LABEL_LENGTH:
                too_long.append((db.parent, idx, len(label), label))
            bad = sorted({ch for ch in label if ch not in allowed})
            if bad:
                bad_charset.append((db.parent, idx, bad, label))

    print(f"Total raw labels: {total}")
    print(f"Missing labels:   {missing}")
    print(f"Too long:         {len(too_long)}")
    print(f"Bad charset:      {len(bad_charset)}")

    if too_long:
        print("\nLongest labels:")
        for db_parent, idx, n, label in sorted(too_long, key=lambda x: x[2], reverse=True)[:20]:
            print(f"  {db_parent.name} #{idx:>6} | len={n:>3} | {label!r}")

    if bad_charset:
        chars = Counter(ch for _, _, bads, _ in bad_charset for ch in bads)
        print("\nUnsupported characters:")
        for ch, count in chars.most_common():
            print(f"  {ch!r}: {count}")
        print("\nFirst bad charset examples:")
        for db_parent, idx, bad, label in bad_charset[:20]:
            print(f"  {db_parent.name} #{idx:>6} | bad={bad} | {label!r}")

inspect_split("train", WORK_DATA_DIR / "train")
inspect_split("val", WORK_DATA_DIR / "val")

## Build the training command

This mirrors your Mac command, with these Colab changes:

- `data.root_dir=/content/parseq_data`
- `trainer.accelerator=gpu` when a GPU is available
- output/checkpoints saved under your Drive `runs/` folder
- your custom `+pretrained_load=auto` option is included

For stock `94_full`, strict loading should succeed. If you later add characters again, leave `PRETRAINED_LOAD="auto"` or set it to `"compatible"`.

In [ ]:
# Create a runnable training script.
from pathlib import Path
import shlex, os, textwrap

RUNS_DIR.mkdir(parents=True, exist_ok=True)

cmd_lines = [
    "HYDRA_FULL_ERROR=1 python train.py",
    "  +experiment=parseq-tiny",
    "  pretrained=parseq-tiny",
    f"  charset={CHARSET}",
    "  'model.charset_test=${model.charset_train}'",
    f"  data.root_dir={shlex.quote(str(WORK_DATA_DIR))}",
    "  data.train_dir=real",
    "  data.remove_whitespace=false",
    "  data.normalize_unicode=false",
    "  data.augment=true",
    f"  data.num_workers={NUM_WORKERS}",
    f"  model.batch_size={BATCH_SIZE}",
    f"  model.max_label_length={MAX_LABEL_LENGTH}",
    f"  trainer.accelerator={ACCELERATOR}",
    f"  trainer.devices={DEVICES}",
    f"  trainer.max_epochs={MAX_EPOCHS}",
    "  trainer.val_check_interval=1.0",
    f"  '+trainer.default_root_dir={str(RUNS_DIR)}'",
    f"  +trainer.log_every_n_steps={LOG_EVERY_N_STEPS}",
    f"  'hydra.run.dir={str(RUNS_DIR)}/${{now:%Y-%m-%d_%H-%M-%S}}'",
    f"  +pretrained_load={PRETRAINED_LOAD}",
    f" '+trainer.enable_progress_bar=false'",
]

script = "#!/usr/bin/env bash\nset -euo pipefail\ncd " + shlex.quote(str(PARSEQ_DIR)) + "\n\n"
script += " \\\n".join(cmd_lines) + "\n"

script_path = Path("/content/train_parseq_colab.sh")
script_path.write_text(script)
script_path.chmod(0o755)

print(script)
print(f"\nWrote: {script_path}")

In [ ]:
# Run training.
!bash /content/train_parseq_colab.sh

In [ ]:
# save/copy/download anything important first!

from google.colab import runtime
runtime.unassign()

## Checkpoints and metrics

Run these after training finishes, or after interrupting a long run.

In [ ]:
# List checkpoints saved to Drive.
!find "{RUNS_DIR}" -name "*.ckpt" -print | sort

In [ ]:
# Print the final scalar metrics from the most recent TensorBoard event file.
from pathlib import Path
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = sorted(
    RUNS_DIR.rglob("events.out.tfevents*"),
    key=lambda p: p.stat().st_mtime
)

if not event_files:
    raise SystemExit(f"No TensorBoard event files found under {RUNS_DIR}")

event_file = event_files[-1]
print(f"Reading TensorBoard event file:\n  {event_file}\n")

ea = EventAccumulator(str(event_file))
ea.Reload()

scalar_tags = ea.Tags().get("scalars", [])
if not scalar_tags:
    raise SystemExit("No scalar metrics found in the TensorBoard event file.")

print("Final logged metrics:")
print("-" * 72)
for tag in sorted(scalar_tags):
    values = ea.Scalars(tag)
    if not values:
        continue
    last = values[-1]
    print(f"{tag:40s} {last.value:.6f}    step={last.step}")
print("-" * 72)

In [ ]:
# TensorBoard viewer.
# Run this cell, then refresh/rerun it after new logs appear.
import os
os.environ["TB_LOGDIR"] = str(RUNS_DIR)
%load_ext tensorboard
%tensorboard --logdir "$TB_LOGDIR"

## Optional: resume from a checkpoint

Paste a `.ckpt` path from the checkpoint-listing cell into `CKPT_PATH`, then run the next cell.

Example:

```python
CKPT_PATH = "/content/drive/MyDrive/Parseq/Training/runs/2026-05-23_12-34-56/checkpoints/last.ckpt"
```

In [ ]:
#@title Optional resume command builder
from pathlib import Path
import shlex

CKPT_PATH = ""  #@param {type:"string"}

if not CKPT_PATH:
    raise SystemExit("Set CKPT_PATH first.")

ckpt = Path(CKPT_PATH)
if not ckpt.exists():
    raise FileNotFoundError(ckpt)

resume_lines = [
    "HYDRA_FULL_ERROR=1 python train.py",
    "  +experiment=parseq-tiny",
    "  pretrained=parseq-tiny",
    f"  charset={CHARSET}",
    "  'model.charset_test=${model.charset_train}'",
    f"  data.root_dir={shlex.quote(str(WORK_DATA_DIR))}",
    "  data.train_dir=real",
    "  data.remove_whitespace=false",
    "  data.normalize_unicode=false",
    "  data.augment=true",
    f"  data.num_workers={NUM_WORKERS}",
    f"  model.batch_size={BATCH_SIZE}",
    f"  model.max_label_length={MAX_LABEL_LENGTH}",
    f"  trainer.accelerator={ACCELERATOR}",
    f"  trainer.devices={DEVICES}",
    f"  trainer.max_epochs={MAX_EPOCHS}",
    "  trainer.val_check_interval=1.0",
    f"  '+trainer.default_root_dir={str(RUNS_DIR)}'",
    f"  +trainer.log_every_n_steps={LOG_EVERY_N_STEPS}",
    f"  'hydra.run.dir={str(RUNS_DIR)}/${{now:%Y-%m-%d_%H-%M-%S}}'",
    f"  +pretrained_load={PRETRAINED_LOAD}",
    f"  ckpt_path={shlex.quote(str(ckpt))}",
]

resume_script = "#!/usr/bin/env bash\nset -euo pipefail\ncd " + shlex.quote(str(PARSEQ_DIR)) + "\n\n"
resume_script += " \\\n".join(resume_lines) + "\n"

resume_script_path = Path("/content/resume_parseq_colab.sh")
resume_script_path.write_text(resume_script)
resume_script_path.chmod(0o755)

print(resume_script)
print(f"\nWrote: {resume_script_path}")

In [ ]:
# Run this only after building the resume script above.
!bash /content/resume_parseq_colab.sh

## Notes

- If the install/import cell gives a NumPy/OpenCV/imgaug import error, use `Runtime → Restart runtime`, then rerun the notebook from the configuration cell downward.
- If the LMDB inspection shows fewer usable validation samples than expected, look first at **Too long** and **Bad charset**.
- Your uploaded `train.py` currently constructs `StochasticWeightAveraging` and passes `callbacks=[checkpoint, swa]`. If you want SWA disabled, edit `train.py` after the patch cell and change the callback list to `callbacks=[checkpoint]`.